In [1]:
from ultralytics import YOLO
from pathlib import Path
import yaml
import os

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\Jadar\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [2]:
print("Current working directory:", os.getcwd())

Current working directory: c:\Users\Jadar\OneDrive\Desktop\Indoor_Object_Detection\exploration


In [3]:
from pathlib import Path
import yaml

yaml_path = Path("../data/data.yaml")

with open(yaml_path, "r", encoding="utf-8") as f:
    data_config = yaml.safe_load(f)

data_config

{'path': '.',
 'train': 'train/images',
 'val': 'valid/images',
 'test': 'test/images',
 'nc': 10,
 'names': ['door',
  'cabinetDoor',
  'refrigeratorDoor',
  'window',
  'chair',
  'table',
  'cabinet',
  'couch',
  'openedDoor',
  'pole']}

In [4]:
print("Number of classes:", data_config["nc"])
print("Class names:")
for i, name in enumerate(data_config["names"]):
    print(i, name)

Number of classes: 10
Class names:
0 door
1 cabinetDoor
2 refrigeratorDoor
3 window
4 chair
5 table
6 cabinet
7 couch
8 openedDoor
9 pole


## Loading a model (in this case I'm using yolov8 nano)

In [5]:
model = YOLO("yolov8n.pt")

## Training the pre-trained yolo nano model on the data I have with some parameters

In [7]:
results = model.train(
    data="../data/data.yaml",
    epochs=10,
    imgsz=640,
    batch=8,
    project="../outputs/runs",
    name="explore_yolov8n",
    device="cpu"
)

Ultralytics 8.4.31  Python-3.13.2 torch-2.11.0+cpu CPU (AMD Ryzen 7 8845HS w/ Radeon 780M Graphics)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../data/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=explore_yolov8n2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=10

## Testing the trained model

In [8]:
metrics = model.val(
    data="../data/data.yaml",
    split="test"
)

Ultralytics 8.4.31  Python-3.13.2 torch-2.11.0+cpu CPU (AMD Ryzen 7 8845HS w/ Radeon 780M Graphics)
Model summary (fused): 73 layers, 3,007,598 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.30.0 ms, read: 51.313.0 MB/s, size: 383.1 KB)
val: Scanning C:\Users\Jadar\OneDrive\Desktop\Indoor_Object_Detection\data\test\labels... 104 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 107/107 428.8it/s 0.2s1s
val: New cache created: C:\Users\Jadar\OneDrive\Desktop\Indoor_Object_Detection\data\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.4it/s 5.2s0.9ss
                   all        107        550       0.61      0.268      0.248      0.126
                  door         25         34      0.391      0.206       0.22     0.0977
           cabinetDoor         42        179      0.562      0.379      0.373      0.137
      refrigeratorDoor          1          2      0.236          1    

## Test results

In [9]:
metrics

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002B307D755E0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,   

## Loading the best params from training in a yolo model 

In [13]:
best_model = YOLO("../runs/outputs/runs/explore_yolov8n2/weights/best.pt")

## Now I check the best params performance (just for the sake of checking)

In [15]:
pred_results = best_model.predict(
    source="../data/test/images",
    save=True,
    project="outputs",
    name="sample_predictions",
    conf=0.25
)


image 1/107 c:\Users\Jadar\OneDrive\Desktop\Indoor_Object_Detection\exploration\..\data\test\images\1003.png: 480x640 1 door, 2 cabinetDoors, 87.4ms
image 2/107 c:\Users\Jadar\OneDrive\Desktop\Indoor_Object_Detection\exploration\..\data\test\images\1014.png: 480x640 1 window, 30.4ms
image 3/107 c:\Users\Jadar\OneDrive\Desktop\Indoor_Object_Detection\exploration\..\data\test\images\1015.png: 480x640 1 cabinetDoor, 27.7ms
image 4/107 c:\Users\Jadar\OneDrive\Desktop\Indoor_Object_Detection\exploration\..\data\test\images\1020.png: 480x640 1 door, 28.1ms
image 5/107 c:\Users\Jadar\OneDrive\Desktop\Indoor_Object_Detection\exploration\..\data\test\images\1021.png: 480x640 1 chair, 29.8ms
image 6/107 c:\Users\Jadar\OneDrive\Desktop\Indoor_Object_Detection\exploration\..\data\test\images\1023.png: 480x640 1 door, 2 windows, 3 chairs, 26.7ms
image 7/107 c:\Users\Jadar\OneDrive\Desktop\Indoor_Object_Detection\exploration\..\data\test\images\1026.png: 480x640 1 door, 1 refrigeratorDoor, 1 window

### Results of this "best model"

In [16]:
print("Prediction images saved.")
print(Path("../outputs/sample_predictions").resolve())

Prediction images saved.
C:\Users\Jadar\OneDrive\Desktop\Indoor_Object_Detection\outputs\sample_predictions
